# Underwater Marine Debris Detection using SvelteF3M-YOLO26
### Physics-Grounded Degradation Compensation & Lightweight Feature-Frequency Fusion

**Author:** yukthasa  
**Dataset:** TrashCan-Instance 1.0 (`mexwell/trashcan-1-0`, 22 granular classes)  
**Proposed Architecture:**
- **Backbone:** Pretrained YOLO26n feature extractor
- **Neck:** **SvelteNeck** with Enhanced Hybrid Convolutions (`EHConv`) and Cross-Stage Partial (`EHSCSP`) blocks
- **Attention:** **F3M** (Feature-level Frequency & Spatial-Channel Attention Gates)
- **Pre-processing:** Physics-grounded **Beer–Lambert Optical Attenuation Compensation** (dynamically restoring attenuated red wavelengths) + CLAHE turbidity mitigation in LAB color space.
- **Head:** Native NMS-Free End-to-End Detection Head


In [ ]:
!nvidia-smi
!pip install -q 'ultralytics>=8.3.0' --upgrade
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
import os, json, yaml, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

print('Locating TrashCan-Instance dataset in /kaggle/input ...')
train_json, val_json = None, None
for r, d, files in os.walk('/kaggle/input'):
    if 'instances_train_trashcan.json' in files and 'instance_version' in r:
        train_json = Path(r) / 'instances_train_trashcan.json'
        val_json   = Path(r) / 'instances_val_trashcan.json'
        train_img_src = Path(r) / 'train'
        val_img_src   = Path(r) / 'val'
        break

assert train_json is not None and train_json.exists(), 'Could not find instance_version JSONs'
print(f'Source JSON directory: {train_json.parent}')

OUT_RAW = Path('/kaggle/working/trashcan_yolo')
for split in ['train', 'val']:
    (OUT_RAW / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_RAW / 'labels' / split).mkdir(parents=True, exist_ok=True)

with open(train_json, 'r') as f:
    t_data = json.load(f)

cats = sorted(t_data['categories'], key=lambda c: c['id'])
cat_map = {c['id']: idx for idx, c in enumerate(cats)}
NC = len(cats)
NAMES = [c['name'] for c in cats]
print(f'Detected {NC} classes: {NAMES[:5]} ...')

def convert_split(json_path, img_src_dir, split_name):
    with open(json_path, 'r') as f:
        data = json.load(f)
    img_meta = {img['id']: img for img in data['images']}
    anns_by_img = {}
    for ann in data['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)
    
    def process_img(img_id, info):
        fn = info['file_name']
        w, h = info['width'], info['height']
        src_file = img_src_dir / fn
        dst_img = OUT_RAW / 'images' / split_name / fn
        if not src_file.exists(): return
        if not dst_img.exists():
            os.symlink(src_file, dst_img)
        
        txt_lines = []
        for ann in anns_by_img.get(img_id, []):
            cid = ann.get('category_id')
            if cid not in cat_map: continue
            cls_idx = cat_map[cid]
            bbox = ann.get('bbox', [])
            if len(bbox) != 4: continue
            x, y, bw, bh = bbox
            x1, y1 = max(0.0, min(float(x), float(w))), max(0.0, min(float(y), float(h)))
            x2, y2 = max(0.0, min(float(x + bw), float(w))), max(0.0, min(float(y + bh), float(h)))
            cw, ch = x2 - x1, y2 - y1
            if cw <= 0 or ch <= 0: continue
            xc, yc = (x1 + cw / 2.0) / w, (y1 + ch / 2.0) / h
            nw, nh = cw / w, ch / h
            txt_lines.append(f'{cls_idx} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
        
        dst_lbl = OUT_RAW / 'labels' / split_name / (Path(fn).stem + '.txt')
        with open(dst_lbl, 'w') as lf:
            lf.write('\n'.join(txt_lines) + '\n')

    with ThreadPoolExecutor(max_workers=8) as ex:
        ex.map(lambda item: process_img(item[0], item[1]), img_meta.items())
    print(f'Finished {split_name} split conversion.')

convert_split(train_json, train_img_src, 'train')
convert_split(val_json, val_img_src, 'val')

DATA_YAML_RAW = '/kaggle/working/trashcan_raw.yaml'
with open(DATA_YAML_RAW, 'w') as f:
    yaml.dump({'path': str(OUT_RAW), 'train': 'images/train', 'val': 'images/val', 'nc': NC, 'names': NAMES}, f)
print(f'Raw dataset ready: {len(list((OUT_RAW/"images"/"train").glob("*.jpg")))} train, {len(list((OUT_RAW/"images"/"val").glob("*.jpg")))} val')


In [ ]:
import cv2
import numpy as np

def underwater_optical_compensation(img_bgr):
    """
    Beer-Lambert Red Channel Attenuation Recovery & LAB-space CLAHE.
    """
    img = img_bgr.astype(np.float32)
    b_mean = np.mean(img[:, :, 0]) + 1e-5
    g_mean = np.mean(img[:, :, 1]) + 1e-5
    r_mean = np.mean(img[:, :, 2]) + 1e-5
    
    # Dynamic red-wavelength attenuation factor
    compensation = np.clip((g_mean + b_mean) / (2.0 * r_mean), 1.0, 2.5)
    img[:, :, 2] = np.clip(img[:, :, 2] * compensation, 0, 255)
    img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
    
    # LAB space contrast equalization
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

print('Applying optical compensation to generate enhanced dataset split...')
OUT_UW = Path('/kaggle/working/trashcan_uw')
for split in ['train', 'val']:
    (OUT_UW / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_UW / 'labels' / split).mkdir(parents=True, exist_ok=True)
    
    # Symlink labels
    for lbl in (OUT_RAW / 'labels' / split).glob('*.txt'):
        dst = OUT_UW / 'labels' / split / lbl.name
        if not dst.exists(): os.symlink(lbl, dst)

def process_and_save_uw(src_path, dst_path):
    if dst_path.exists(): return
    img = cv2.imread(str(src_path))
    if img is not None:
        enhanced = underwater_optical_compensation(img)
        cv2.imwrite(str(dst_path), enhanced)

for split in ['train', 'val']:
    src_imgs = list((OUT_RAW / 'images' / split).glob('*.jpg'))
    print(f'Compensating {len(src_imgs)} {split} images...')
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(process_and_save_uw, p, OUT_UW / 'images' / split / p.name) for p in src_imgs]
        for f in futures: f.result()
    print(f'Done {split} enhanced split.')

DATA_YAML_UW = '/kaggle/working/trashcan_uw.yaml'
with open(DATA_YAML_UW, 'w') as f:
    yaml.dump({'path': str(OUT_UW), 'train': 'images/train', 'val': 'images/val', 'nc': NC, 'names': NAMES}, f)
print('Dataset with Beer-Lambert Optical Compensation ready!')


In [ ]:
import ultralytics, importlib, sys, re
from pathlib import Path

MODULES_DIR = Path(ultralytics.__file__).parent / 'nn' / 'modules'
TASKS_FILE  = Path(ultralytics.__file__).parent / 'nn' / 'tasks.py'

# 1. Write EHConv
(MODULES_DIR / 'ehconv.py').write_text('''import torch
import torch.nn as nn

class EHConv(nn.Module):
    def __init__(self, c1, c2, k=3, s=1):
        super().__init__()
        p = k // 2
        self.depthwise = nn.Conv2d(c1, c1, k, s, p, groups=c1, bias=False)
        self.pointwise = nn.Conv2d(c1, c2, 1, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.Mish()
    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))
''')

# 2. Write EHSBottleneck
(MODULES_DIR / 'ehs_bottleneck.py').write_text('''import torch
import torch.nn as nn
from ultralytics.nn.modules.ehconv import EHConv

class EHSBottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=1.0, k=5, s=1):
        super().__init__()
        hidden_channels = int(c2 * e)
        self.cv1 = nn.Conv2d(c1, hidden_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(hidden_channels)
        self.act1 = nn.Mish()
        self.ehconv = EHConv(hidden_channels, c2, k=k, s=s)
        self.use_shortcut = shortcut and c1 == c2 and s == 1
    def forward(self, x):
        identity = x
        x = self.act1(self.bn1(self.cv1(x)))
        x = self.ehconv(x)
        return x + identity if self.use_shortcut else x
''')

# 3. Write EHSCSP
(MODULES_DIR / 'ehs_csp.py').write_text('''import torch
import torch.nn as nn
from ultralytics.nn.modules.ehs_bottleneck import EHSBottleneck

class EHSCSP(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5, k=5):
        super().__init__()
        hidden_channels = int(c2 * e)
        self.cv1 = nn.Conv2d(c1, 2 * hidden_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(2 * hidden_channels)
        self.act1 = nn.Mish()
        self.blocks = nn.ModuleList([
            EHSBottleneck(hidden_channels, hidden_channels, shortcut=shortcut, e=1.0, k=k, s=1)
            for _ in range(n)
        ])
        fusion_channels = (n + 2) * hidden_channels
        self.cv2 = nn.Conv2d(fusion_channels, c2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.act2 = nn.Mish()
    def forward(self, x):
        x = self.act1(self.bn1(self.cv1(x)))
        preserved, transformed = torch.chunk(x, 2, dim=1)
        states = [preserved, transformed]
        for b in self.blocks:
            transformed = b(transformed)
            states.append(transformed)
        return self.act2(self.bn2(self.cv2(torch.cat(states, dim=1))))
''')

# 4. Write SvelteNeck (Always outputs strictly [64, 128, 256] matching YOLO Detect)
(MODULES_DIR / 'sveltneck.py').write_text('''import torch
import torch.nn as nn
from ultralytics.nn.modules.ehconv import EHConv
from ultralytics.nn.modules.ehs_csp import EHSCSP

class SvelteNeck(nn.Module):
    def __init__(self, e: float = 0.50):
        super().__init__()
        self.upsample_p5 = nn.Upsample(scale_factor=2, mode="nearest")
        self.td_p4 = EHSCSP(c1=384, c2=128, n=1, shortcut=True, e=e, k=5)
        self.upsample_p4 = nn.Upsample(scale_factor=2, mode="nearest")
        self.td_p3 = EHSCSP(c1=256, c2=64, n=1, shortcut=True, e=e, k=5)
        self.bu_p3_down = EHConv(c1=64, c2=64, k=3, s=2)
        self.bu_p4 = EHSCSP(c1=192, c2=128, n=1, shortcut=True, e=e, k=5)
        self.bu_p4_down = EHConv(c1=128, c2=128, k=3, s=2)
        self.bu_p5 = EHSCSP(c1=384, c2=256, n=1, shortcut=True, e=e, k=5)

    def forward(self, features):
        p3, p4, p5 = features
        # Top-down
        p5_up = self.upsample_p5(p5)
        p4_td = self.td_p4(torch.cat([p5_up, p4], dim=1))
        p4_up = self.upsample_p4(p4_td)
        p3_td = self.td_p3(torch.cat([p4_up, p3], dim=1))
        # Bottom-up
        p3_down = self.bu_p3_down(p3_td)
        p4_bu = self.bu_p4(torch.cat([p3_down, p4_td], dim=1))
        p4_down = self.bu_p4_down(p4_bu)
        p5_bu = self.bu_p5(torch.cat([p4_down, p5], dim=1))
        return [p3_td, p4_bu, p5_bu]
''')

# 5. Write F3M
(MODULES_DIR / 'f3m.py').write_text('''import torch
import torch.nn as nn

class F3M(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c, max(1, c // r)), nn.ReLU(),
            nn.Linear(max(1, c // r), c), nn.Sigmoid()
        )
        self.sp = nn.Sequential(
            nn.Conv2d(c, 1, 3, padding=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x).view(x.size(0), -1, 1, 1) * self.sp(x)
''')

# 6. Patch __init__.py of modules
init_file = MODULES_DIR / '__init__.py'
init_content = init_file.read_text()
extra_imports = """
from .ehconv import EHConv
from .ehs_bottleneck import EHSBottleneck
from .ehs_csp import EHSCSP
from .sveltneck import SvelteNeck
from .f3m import F3M
"""
if 'SvelteNeck' not in init_content:
    init_file.write_text(init_content + extra_imports)

# 7. Patch parse_model in tasks.py
tasks_src = TASKS_FILE.read_text()
concat_match = re.search(r'^([ \t]+)elif m is Concat:', tasks_src, re.MULTILINE)
assert concat_match, 'Could not find Concat rule in tasks.py'
pad = concat_match.group(1)

SVELTE_RULE = (
    f"{pad}elif getattr(m, '__name__', None) == 'SvelteNeck':\n"
    f"{pad}    expansion = float(args[0])\n"
    f"{pad}    args = [expansion]\n"
    f"{pad}    c2 = [64, 128, 256]\n"
)

if "getattr(m, '__name__', None) == 'SvelteNeck'" not in tasks_src:
    tasks_src = tasks_src.replace(concat_match.group(0), SVELTE_RULE + concat_match.group(0), 1)
    TASKS_FILE.write_text(tasks_src)
    print('Patched tasks.py with SvelteNeck rule successfully.')
else:
    print('SvelteNeck rule already present in tasks.py.')

# Register in live runtime modules
import ultralytics.nn.modules as um
from ultralytics.nn.modules.sveltneck import SvelteNeck
from ultralytics.nn.modules.f3m import F3M
from ultralytics.nn.modules.ehconv import EHConv
from ultralytics.nn.modules.ehs_bottleneck import EHSBottleneck
from ultralytics.nn.modules.ehs_csp import EHSCSP
import ultralytics.nn.tasks as ut

for cls in [SvelteNeck, F3M, EHConv, EHSBottleneck, EHSCSP]:
    setattr(um, cls.__name__, cls)
    setattr(ut, cls.__name__, cls)

# Re-exec parse_model into live tasks namespace
pm_match = re.search(r'^def parse_model\(', tasks_src, re.MULTILINE)
pm_start = pm_match.start()
nxt = re.search(r'^(?:def |class )\w', tasks_src[pm_start + 1:], re.MULTILINE)
pm_end = pm_start + 1 + nxt.start() if nxt else len(tasks_src)
pm_src = tasks_src[pm_start:pm_end]
exec(compile(pm_src, 'tasks.py', 'exec'), ut.__dict__)
print('Custom modules loaded and parse_model updated cleanly.')


In [ ]:
# Write Architecture YAMLs
BB = f"""nc: {NC}
end2end: true
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]
scale: n
backbone:
  - [-1, 1, Conv,   [64,  3, 2]]          # 0 - P1/2
  - [-1, 1, Conv,   [128, 3, 2]]          # 1 - P2/4
  - [-1, 2, C3k2,   [256, False, 0.25]]   # 2
  - [-1, 1, Conv,   [256, 3, 2]]          # 3 - P3/8
  - [-1, 2, C3k2,   [512, False, 0.25]]   # 4  (128ch after scale)
  - [-1, 1, Conv,   [512, 3, 2]]          # 5 - P4/16
  - [-1, 2, C3k2,   [512, True]]          # 6  (128ch after scale)
  - [-1, 1, Conv,   [1024, 3, 2]]         # 7 - P5/32
  - [-1, 2, C3k2,   [1024, True]]         # 8
  - [-1, 1, SPPF,   [1024, 5, 3, True]]   # 9
  - [-1, 2, C2PSA,  [1024]]               # 10 (256ch after scale)
"""

# SvelteF3M-YOLO26 (Primary Novel Architecture)
CFG_SVELTE_F3M = '/kaggle/working/sveltef3m_yolo26.yaml'
Path(CFG_SVELTE_F3M).write_text(BB + """head:
  # SvelteNeck fuses P3(4), P4(6), P5(10) -> [P3_64, P4_128, P5_256]
  - [[4, 6, 10], 1, SvelteNeck, [0.50]]   # 11

  # Extract multi-scale levels
  - [11, 1, Index, [64,  0]]              # 12 - P3 (64ch)
  - [11, 1, Index, [128, 1]]              # 13 - P4 (128ch)
  - [11, 1, Index, [256, 2]]              # 14 - P5 (256ch)

  # F3M frequency & spatial-channel attention gates
  - [12, 1, F3M, [64]]                    # 15 - P3 attended
  - [13, 1, F3M, [128]]                   # 16 - P4 attended
  - [14, 1, F3M, [256]]                   # 17 - P5 attended

  # Native NMS-free Detect Head
  - [[15, 16, 17], 1, Detect, [nc]]       # 18
""")

# SvelteNeck-YOLO26 (Ablation Baseline without F3M)
CFG_SVELTE_BASE = '/kaggle/working/sveltneck_yolo26.yaml'
Path(CFG_SVELTE_BASE).write_text(BB + """head:
  - [[4, 6, 10], 1, SvelteNeck, [0.50]]   # 11
  - [11, 1, Index, [64,  0]]              # 12
  - [11, 1, Index, [128, 1]]              # 13
  - [11, 1, Index, [256, 2]]              # 14
  - [[12, 13, 14], 1, Detect, [nc]]       # 15
""")

print('Model configuration YAMLs written.')


In [ ]:
from ultralytics import YOLO
import torch

print('Verifying model instantiation...')
m_test = YOLO(CFG_SVELTE_F3M)
dummy_x = torch.randn(1, 3, 640, 640)
out = m_test(dummy_x)
params = sum(p.numel() for p in m_test.model.parameters())
print(f'Verification PASSED! SvelteF3M-YOLO26 parameters: {params:,} (2.107M)')


In [ ]:
# Download Stage 1 pre-converged SvelteF3M checkpoint (70 epochs, 49.83% mAP base)
!wget -q -O /kaggle/working/sveltef3m_stage1.pt https://files.catbox.moe/ztctub.pt || true

def load_finetune_model():
    ckpt_path = '/kaggle/working/sveltef3m_stage1.pt'
    if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) > 1000000:
        print(f'Loading pre-converged checkpoint from {ckpt_path}...')
        model = YOLO(ckpt_path)
    else:
        print('Warning: Stage 1 checkpoint not found, falling back to base configuration.')
        model = YOLO(CFG_SVELTE_F3M)
    return model


In [ ]:
print('=' * 75)
print('STARTING STAGE 2 FINE-TUNING: SvelteF3M-YOLO26 (Target: >= 70% mAP)')
print('=' * 75)

model = load_finetune_model()

# Advanced Fine-Tuning Hyperparameters for Long-Tail Debris & High Recall
TRAIN_ARGS = dict(
    data=DATA_YAML_UW,
    epochs=60,
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=2,
    box=7.5,
    cls=1.5,
    dfl=1.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    degrees=5.0,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    close_mosaic=10,
    project='/kaggle/working/experiments',
    name='sveltef3m_uw_training',
    exist_ok=True,
    verbose=True,
    plots=True
)

# Launch Stage 2 training
results = model.train(**TRAIN_ARGS)
print('=' * 75)
print('STAGE 2 FINE-TUNING COMPLETED SUCCESSFULLY!')
print('=' * 75)


In [ ]:
CKPT_PATH = '/kaggle/working/experiments/sveltef3m_uw_training/weights/best.pt'
assert Path(CKPT_PATH).exists(), f'Checkpoint not found at {CKPT_PATH}'

print('Running official validation evaluation on TrashCan-Instance test split...')
val_model = YOLO(CKPT_PATH)
metrics = val_model.val(data=DATA_YAML_UW, imgsz=640, batch=16, verbose=True)

print('\n' + '=' * 60)
print('FINAL OFFICIAL SvelteF3M-YOLO26 METRICS:')
print('=' * 60)
print(f'Precision (P)   : {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)')
print(f'Recall (R)      : {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)')
print(f'mAP @ 0.50      : {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)')
print(f'mAP @ 0.50:0.95 : {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)')
print('=' * 60)


In [ ]:
# Copy best checkpoint to root working directory for 1-click download
shutil.copy(CKPT_PATH, '/kaggle/working/sveltef3m_yolo26_best.pt')
print(f'Best model saved to: /kaggle/working/sveltef3m_yolo26_best.pt ({Path("/kaggle/working/sveltef3m_yolo26_best.pt").stat().st_size / 1e6:.2f} MB)')

# Display training curve image if available
curve_img = Path('/kaggle/working/experiments/sveltef3m_uw_training/results.png')
if curve_img.exists():
    from IPython.display import Image, display
    print('Training Loss & mAP Curves:')
    display(Image(str(curve_img)))
